# Tratativas

### Imports:

In [ ]:
%pip install pandas
%pip install openpyxl

In [ ]:
import pandas as pd

### Criação do banco de dados .csv

In [ ]:
nome_arquivo_excel = '../dados/base_quedas_AACD_anonimizada.xlsx'
nome_arquivo_csv = '../dados/base_quedas_AACD_anonimizada.csv'

try:
    # Carregar as duas abas do Excel
    print("Lendo as abas do Excel...")
    df_60_mais = pd.read_excel(nome_arquivo_excel, sheet_name='60+')
    df_60_menos = pd.read_excel(nome_arquivo_excel, sheet_name='60-')

    # Função para juntar as linhas de 2 em 2
    def consertar_linhas_duplas(df):
        # Agrupa de 2 em 2 linhas e pega o primeiro valor não nulo
        df_agrupado = df.groupby(df.index // 2).first()
        return df_agrupado

    # Aplicar a correção nas duas tabelas
    print("Corrigindo o cadastro feito em duas linhas...")
    df_60_mais_corrigido = consertar_linhas_duplas(df_60_mais)
    df_60_menos_corrigido = consertar_linhas_duplas(df_60_menos)

    # Juntar as duas tabelas em uma só
    print("Unindo as duas tabelas...")
    df_final = pd.concat([df_60_mais_corrigido, df_60_menos_corrigido], ignore_index=True)

    # ---------------------------------------------------------
    print("Corrigindo a numeração dos IDs...")
    df_final['ID'] = range(1, len(df_final) + 1)
    # ---------------------------------------------------------

    # Salvar o resultado no arquivo .csv
    df_final.to_csv(nome_arquivo_csv, index=False, encoding='utf-8-sig')
    
    print(f"Sucesso! Arquivo '{nome_arquivo_csv}' criado com IDs sequenciais.")

except FileNotFoundError:
    print(f"Erro: O arquivo '{nome_arquivo_excel}' não foi encontrado na pasta.")
except Exception as e:
    print(f"Ocorreu um erro: {e}")

### Correções do nome das colunas:

In [ ]:
df = pd.read_csv('../dados/base_quedas_AACD_anonimizada.csv')

colunas_com_interrogacao = [
    'Clínica/Dx',
    'Aditamento 10m',
    'Status funcional',
    'Diagnóstico Funcional',
]
for coluna in colunas_com_interrogacao:
    df[coluna] = df[coluna].astype('string').str.replace('?', '', regex=False)

colunas = df.columns.tolist()
print(colunas)

In [ ]:
df.rename(columns={'Polifarmácia?': 'Polifarmácia'}, inplace=True)
print(df.columns)

In [ ]:
df.rename(columns={'Idade ': 'Idade'}, inplace=True)
print(df.columns)

### Correções nas colunas

- Apaga qualquer espaço em branco antes ou depois da palavra, transforma 'sim ' em 'sim'.

In [ ]:
df['Polifarmácia'] = (
    df['Polifarmácia']
    .astype('string')
    .str.strip()
    .replace({'': pd.NA})
    .fillna('não')
    .astype('category')
)

print(df['Polifarmácia'].dtype)
print(df['Polifarmácia'].value_counts())

- Remove a marca "?" antes da conversão numérica. Idades inválidas permanecem ausentes, sem imputação.

In [ ]:
df['Idade'] = df['Idade'].astype('string').str.replace('?', '', regex=False).str.strip()

In [ ]:
df['Idade'] = pd.to_numeric(df['Idade'], errors='coerce').astype('Int64')
# Calcula a média das idades (desconsiderando valores nulos)
df['Idade'] = pd.to_numeric(df['Idade'], errors='coerce').astype('Int64')
media_idade = df['Idade'].mean()
media_idade = int(round(media_idade))

df.loc[df['ID'] == 275, 'Idade'] = media_idade

#### Correções de tipagem

In [ ]:
df.tail(1)

In [ ]:
df.dtypes

In [ ]:
df = df.rename(columns={'Cognição         (10-CS)': 'Cogniçao (10-CS)'})

In [ ]:
df['Reação (blazepod)'] = pd.to_numeric(df['Reação (blazepod)'], errors='coerce').astype('Int64')
df['Reação (blazepod) DT'] = pd.to_numeric(df['Reação (blazepod) DT'], errors='coerce').astype('Int64')
df['CIF equilíbrio'] = pd.to_numeric(df['CIF equilíbrio'], errors='coerce').astype('Int64')
df['Ponte'] = pd.to_numeric(df['Ponte'], errors='coerce').astype('Int64')
df['Sentar/levantar 30s'] = pd.to_numeric(df['Sentar/levantar 30s'], errors='coerce').astype('Int64')
df['Nº quedas ao chão'] = pd.to_numeric(df['Nº quedas ao chão'], errors='coerce').astype('Int64')
df['Cogniçao (10-CS)'] = pd.to_numeric(df['Cogniçao (10-CS)'], errors='coerce').astype('Int64')
df['Cognição (MOCA)'] = pd.to_numeric(df['Cognição (MOCA)'], errors='coerce').astype('Int64')
df['Dinamômetro'] = pd.to_numeric(df['Dinamômetro'], errors='coerce').round().astype('Int64')

### Cálculo da velocidade no teste de 10 metros

A coluna `10 metros` registra o tempo em segundos. A velocidade é calculada em metros por segundo (m/s) pela distância de 10 metros dividida pelo tempo. Tempos ausentes ou não positivos geram velocidade ausente.

In [ ]:
df['10 metros'] = pd.to_numeric(df['10 metros'], errors='coerce')
df['Velocidade (m/s)'] = 10 / df['10 metros'].where(df['10 metros'] > 0)

In [ ]:
df['Sexo'] = df['Sexo'].astype('category')
df['Posição equilíbrio'] = df['Posição equilíbrio'].astype('category')
df['Panturrilha D'] = pd.to_numeric(df['Panturrilha D'], errors='coerce').astype('float')

In [ ]:
df.isnull().sum()

### Tratamento conservador das colunas de suporte e status

Padroniza caixa e espaços, converte células vazias e o valor "." em ausentes, sem agrupar descrições diferentes.

In [ ]:
colunas_texto_livre = [
    'Suporte social (terapêutico)',
    'Suporte social (domiciliar)',
    'Status funcional',
]

for coluna in colunas_texto_livre:
    df[coluna] = (
        df[coluna]
        .astype('string')
        .str.strip()
        .str.replace(r'\s+', ' ', regex=True)
        .str.lower()
        .replace(['', '.'], pd.NA)
    )

colunas_suporte = [
    'Suporte social (terapêutico)',
    'Suporte social (domiciliar)',
]
df[colunas_suporte] = df[colunas_suporte].fillna('não informado')

In [ ]:
pd.DataFrame({
    'valores_distintos': df[colunas_texto_livre].nunique(dropna=True),
    'ausentes': df[colunas_texto_livre].isna().sum(),
})

### Salva as alterações diretamente no arquivo .csv

Facilita a visualização diretamente no arquivo

In [ ]:
df['Polifarmácia'] = (
    df['Polifarmácia']
    .astype('string')
    .str.strip()
    .replace({'': pd.NA, 'nan': pd.NA, 'NaN': pd.NA, 'None': pd.NA, 'null': pd.NA, '<NA>': pd.NA})
    .fillna('não')
    .astype('category')
)

df.to_csv('../dados/base_quedas_AACD_anonimizada.csv', index=False, encoding='utf-8-sig')
print("Arquivo salvo com sucesso!")

##### Tratamento de nulos

In [ ]:
df['Queda último mês'].isnull().sum()

In [ ]:
df['Queda último mês'] = df['Queda último mês'].astype(str).str.strip()

df['Queda último mês'] = df['Queda último mês'].replace({
    '0': 'não',
    '2': 'sim',
    '': 'não informado',
    'nan': 'não informado',
    'NaN': 'não informado',
    'null': 'não informado',
    'None': 'não informado'
})

df['Queda último mês'] = df['Queda último mês'].fillna('não informado')
df['Queda último mês'] = df['Queda último mês'].astype('category')

In [ ]:
print(df['Queda último mês'].isna().sum())

print(df['Queda último mês'].value_counts(dropna=False))

In [ ]:
df['Nº quedas ao chão'] = df['Nº quedas ao chão'].astype(str).str.strip()

df['Nº quedas ao chão'] = df['Nº quedas ao chão'].replace({
    '': 'não informado',
    'nan': 'não informado',
    'NaN': 'não informado',
    'null': 'não informado',
    'None': 'não informado'
})

df['Nº quedas ao chão'] = df['Nº quedas ao chão'].fillna('não informado')

##### Padronizar os registros

In [ ]:
df['Clínica/Dx'] = df['Clínica/Dx'].astype('string').str.strip().str.upper()

# Dicionário de substituição dos diagnósticos
diagnosticos = {
    'AMP': 'AMP - Amputação',
    'AMP BILATERAL': 'AMP Bilateral - Amputação bilateral',
    'AMP LEA': 'AMP LEA - Amputação e Lesão Encefálica Adquirida',
    'AVC': 'AVC - Acidente Vascular Cerebral',
    'DNM': 'DNM - Doença Neuromuscular',
    'ELA': 'ELA - Esclerose Lateral Amiotrófica',
    'LEA': 'LEA - Lesão Encefálica Adquirida',
    'LM': 'LM - Lesão Medular',
    'PARKISON': 'Parkinson',
    'PARKINSON': 'Parkinson',
    'PC': 'PC - Paralisia Cerebral',
    'POLIO': 'Poliomielite',
    'POLIO?': 'Poliomielite',
    'MFC': 'MFC - Malformações Congênitas',
    'Mielo': 'Mielomeningocele',
    'EM': 'EM - Esclerose Múltipla',
    'LEA/LM': 'LEA/LM - Lesão Encefálica Adquirida e Lesão Medular'
}
df['Clínica/Dx'] = df['Clínica/Dx'].replace(diagnosticos)

### Tratativa da coluna status funcional

In [ ]:
# vendo quantidade de valores nulos na coluna 'Status funcional'
df['Status funcional'].isnull().sum()

# Substitui os vazios (e o 'nan' que virou texto) por "não informado"
df['Status funcional'] = df['Status funcional'].fillna('não informado')

# substitutindo o texto cadeira de rodas por cadeirante
df['Status funcional'] = df['Status funcional'].replace('cadeira de rodas', 'cadeirante')
df['Status funcional'] = df['Status funcional'].replace('deslocamento por cr', 'cadeirante')
df['Status funcional'] = df['Status funcional'].replace('marca comunitária', 'marcha comunitária')

#padronizando os textos para minúsculo
df['Status funcional'] = df['Status funcional'].str.lower()

#corrigindo os textos com erros de digitação
df['Status funcional'] = df['Status funcional'].replace({
    'marcha comunitária i': 'marcha comunitária independente',
    'cadeira de rodas ': 'cadeirante',
    'marhca comunitária com andador': 'marcha comunitária com andador',
    'marhca comunitária com bastão e supervisão': 'marcha comunitária com bastão e supervisão',
    'marcha comunitária i': 'marcha comunitária independente',
    
})

# removendo os '?' que ficaram no meio do texto
df['Status funcional'] = df['Status funcional'].str.replace('?', '', regex=False)

print(df['Status funcional'].value_counts())



In [ ]:
#visualizando os a colunas 'status funcional'
df[['Status funcional']]


In [ ]:
# substituindoas as linhas vazias da coluna 'Diagnostico Funcional' por "não informado" adaptando o código para lidar com linhas com espaços em branco e valores nulos
df['Diagnóstico Funcional'] = df['Diagnóstico Funcional'].str.strip().fillna('não informado')
print(df['Diagnóstico Funcional'].value_counts())

#corrigindo os textos com erros de digitação
df['Diagnóstico Funcional'] = df['Diagnóstico Funcional'].replace({
    'Dupla hemi ': 'Dupla Hemi'
})

In [ ]:
print(df['Nº quedas ao chão'].dtype)
print(df['Nº quedas ao chão'].value_counts())

#mudando a coluna 'Nº quedas ao chão' para o tipo inteiro, para que possamos fazer análises estatísticas
df['Nº quedas ao chão'] = pd.to_numeric(df['Nº quedas ao chão'], errors='coerce').astype('Int64')
